### Horizontal and vertical cuts of an envelope pattern

Load one envelope JSON, slice a horizontal and a vertical cut out of it, plot both.

In [ ]:
import plotly.graph_objects as go
import xarray as xr

from eas_3d_pattern import SAMPLE_JSON, AntennaPattern

antenna_pattern = AntennaPattern(SAMPLE_JSON[2])  # this sample declares Pattern_Type "Traffic Envelope"
print(f"{antenna_pattern.antenna_model} | {antenna_pattern.pattern_type} | {antenna_pattern.frequency_hz / 1e6:.0f} MHz")

The envelope data sits in `antenna_pattern.pattern`, an xarray dataset on a (Theta, Phi) grid. We take the normalized total power `P_tp_dB`.

A cut is one `sel()` on the dataset: fix Theta and the horizontal cut remains, fix Phi and the vertical cut remains. Both go through the beam peak.

In [ ]:
theta_peak, phi_peak = antenna_pattern.find_peak_coordinates()
power = antenna_pattern.pattern["P_tp_dB"]

horizontal_cut = power.sel(Theta=theta_peak, method="nearest")  # left over dimension: Phi
vertical_cut = power.sel(Phi=phi_peak, method="nearest")  # left over dimension: Theta

print(f"peak at Theta={theta_peak}deg, Phi={phi_peak}deg")

A minimal plot function. The one remaining dimension of the cut becomes the x axis.

In [ ]:
def plot_cut(cut: xr.DataArray, title: str) -> None:
    """Plot a single pattern cut as a line.

    Args:
        cut (xr.DataArray): One dimensional cut, indexed by Theta or Phi.
        title (str): Figure title.
    """
    angle_name = cut.dims[0]
    fig = go.Figure(go.Scatter(x=cut[angle_name].values, y=cut.values, mode="lines"))
    fig.update_layout(
        title=title,
        xaxis_title=f"{angle_name} [deg]",
        yaxis_title="Normalized power [dB]",
        yaxis_range=[-40, 2],
        width=850,
        height=400,
    )
    fig.show()

In [ ]:
plot_cut(horizontal_cut, f"Horizontal cut at Theta = {float(horizontal_cut.Theta):.0f}deg")
plot_cut(vertical_cut, f"Vertical cut at Phi = {float(vertical_cut.Phi):.0f}deg")